# Topic Modeling

## Load the dataset

In [7]:
import pandas as pd

exploratory_df = pd.read_csv("dataset/processed/exploratory-packages-feature-engineering.csv")
confirmatory_df = pd.read_csv("dataset/processed/confirmatory-packages-feature-engineering.csv")
holdout_df = pd.read_csv("dataset/processed/holdout-packages-feature-engineering.csv")

exploratory_df.head()

,package_id,created_at,test_week,clickability_test_id,headline,eyecatcher_id,impressions,clicks,ctr,ctr_demeaned,...,neg,neu,pos,compound,created_at_dayofweek,created_at_hourofday,test_group_size,read_flesch,read_coleman,specificity_tfidf
0,62,2014-11-20 14:57:52.478,201446,546e009a9ad54ec65b00004b,What They Learned From The Scientist Was Terri...,546c7f2dbadeb5788700000a,4594,51,0.011101,0.111516,...,0.144,0.856,0.000,-0.3291,3,14,6,80.782500,8.093333,0.444616
1,84,2014-11-20 14:54:18.780,201446,546e009a9ad54ec65b00004b,A Science Guy Helps 3 Dudes From America Under...,546c7f2dbadeb5788700000a,4571,58,0.012689,0.682108,...,0.000,0.809,0.191,0.3818,3,14,6,59.682143,10.257143,0.370851
2,95,2014-11-20 15:04:49.517,201446,546e009a9ad54ec65b00004b,He Sat Them Down And Told Them About An Immine...,546c7f2dbadeb5788700000a,4601,27,0.005868,-1.769714,...,0.187,0.813,0.000,-0.5994,3,15,6,80.465000,6.077778,0.398801
3,100,2014-11-20 15:13:36.266,201446,546e009a9ad54ec65b00004b,"The 3 Of Them Needed To See It In Person, And ...",546c7f2dbadeb5788700000a,4567,63,0.013795,1.079669,...,0.124,0.720,0.156,0.2023,3,15,6,80.777143,3.228571,0.445514
4,102,2014-11-20 15:15:25.697,201446,546e009a9ad54ec65b00004b,"They May Not Be The Most Handsome Dudes, But T...",546c7f2dbadeb5788700000a,4524,44,0.009726,-0.382964,...,0.000,0.655,0.345,0.7812,3,15,6,92.965000,5.875000,0.497006


## BERTopic

I decided to use the `all-MiniLM-L6-v2` model for the embeddings as it's a simple model for this task with a good performance track and recommended by the library documentation.

In [8]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

exploratory_headlines = exploratory_df["headline"].fillna("").tolist()
confirmatory_headlines = confirmatory_df["headline"].fillna("").tolist()
holdout_headlines = holdout_df["headline"].fillna("").tolist()

# Create the SentenceTransformer embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Fit the BERTTopic model
# Set min_topic_size so only topics with at least 80 headlines are considered
# For IF-IDF I use unigrams and bigrams
topic_model = BERTopic(
    embedding_model=embedding_model,
    n_gram_range=(1, 2),
    min_topic_size=60,
    top_n_words=20,
    verbose=True,
)

exploratory_topics, exploratory_probs = topic_model.fit_transform(exploratory_headlines)

exploratory_df["topic_bertopic"] = exploratory_topics

# Use the fitted model to assign topics to confirmatory and holdout sets
confirmatory_headlines = confirmatory_df["headline"].fillna("").tolist()
holdout_headlines = holdout_df["headline"].fillna("").tolist()
confirmatory_topics, confirmatory_probs = topic_model.transform(confirmatory_headlines)
holdout_topics, holdout_probs = topic_model.transform(holdout_headlines)
confirmatory_df["topic_bertopic"] = confirmatory_topics
holdout_df["topic_bertopic"] = holdout_topics

exploratory_df[["headline", "topic_bertopic"]].head()


2025-12-15 17:27:21,626 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/376 [00:00<?, ?it/s]

2025-12-15 17:27:34,208 - BERTopic - Embedding - Completed ✓
2025-12-15 17:27:34,209 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-15 17:27:36,606 - BERTopic - Dimensionality - Completed ✓
2025-12-15 17:27:36,610 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-15 17:27:37,067 - BERTopic - Cluster - Completed ✓
2025-12-15 17:27:37,075 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-15 17:27:37,351 - BERTopic - Representation - Completed ✓


Batches:   0%|          | 0/1795 [00:00<?, ?it/s]

2025-12-15 17:28:35,740 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-12-15 17:28:44,486 - BERTopic - Dimensionality - Completed ✓
2025-12-15 17:28:44,487 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-12-15 17:28:46,729 - BERTopic - Cluster - Completed ✓


Batches:   0%|          | 0/390 [00:00<?, ?it/s]

2025-12-15 17:28:59,728 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2025-12-15 17:29:01,373 - BERTopic - Dimensionality - Completed ✓
2025-12-15 17:29:01,373 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2025-12-15 17:29:01,845 - BERTopic - Cluster - Completed ✓


,headline,topic_bertopic
0,What They Learned From The Scientist Was Terri...,6
1,A Science Guy Helps 3 Dudes From America Under...,6
2,He Sat Them Down And Told Them About An Immine...,-1
3,"The 3 Of Them Needed To See It In Person, And ...",-1
4,"They May Not Be The Most Handsome Dudes, But T...",-1


In [9]:
# Print the number of headlines per topic
print(f"Exploratory set: Number of headlines per topic ({len(exploratory_df)} headlines):")
print(exploratory_df["topic_bertopic"].value_counts().head(20))

print(f"Confirmatory set: Number of headlines per topic ({len(confirmatory_df)} headlines):")
print(confirmatory_df["topic_bertopic"].value_counts().head(20))

print(f"Holdout set: Number of headlines per topic ({len(holdout_df)} headlines):")
print(holdout_df["topic_bertopic"].value_counts().head(20))


Exploratory set: Number of headlines per topic (12010 headlines):
topic_bertopic
-1     6675
 0      864
 1      490
 2      340
 3      334
 4      283
 5      272
 6      188
 7      181
 8      180
 9      179
 10     156
 11     143
 12     142
 13     126
 14     125
 15     119
 16     108
 17     108
 18     100
Name: count, dtype: int64
Confirmatory set: Number of headlines per topic (57413 headlines):
topic_bertopic
-1     32089
 0      3848
 1      2370
 3      1805
 4      1744
 2      1648
 5      1308
 6      1000
 7       949
 9       824
 8       800
 14      726
 12      665
 16      656
 13      612
 15      545
 10      544
 11      518
 20      498
 17      472
Name: count, dtype: int64
Holdout set: Number of headlines per topic (12474 headlines):
topic_bertopic
-1     6922
 0      794
 1      520
 3      437
 2      392
 4      384
 5      264
 7      193
 9      181
 14     166
 6      164
 8      164
 20     150
 25     140
 16     139
 11     132
 12     121
 24 

Around half the headlines don't have a topic assigned. This is ok, we only care about the topics we can really identify rather than noise.

The good thing is that the topics discovered using the exploratory set were fitted to the confirmatory and holdout sets in similar proportions.

### Topics Analysis

In [10]:
# Inspect the discovered topics

# Overview of the 20 most common topics
topic_info = topic_model.get_topic_info()
topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,6675,-1_the_to_you_of,"[the, to, you, of, and, it, this, in, is, that...",[If You Ever Wondered What People Who Don't Be...
1,0,864,0_she_her_to_the,"[she, her, to, the, it, and, was, in, mom, but...",[OMG. They Tell Her That Her Words Are BS Righ...
2,1,490,1_kids_to_this_the,"[kids, to, this, the, teachers, these, kid, of...",[Leave It To A Little Kid To Come Up With A Cr...
3,2,340,2_food_eat_you_it,"[food, eat, you, it, the, that, to, fast food,...",[Fast Food Usually Grosses Me Out. But This Vi...
4,3,334,3_black_white_racist_race,"[black, white, racist, race, people, racism, i...",[If You're Wondering How To Talk About Black P...
5,4,283,4_gay_straight_marriage_gay marriage,"[gay, straight, marriage, gay marriage, the, p...",[The Most Sexist And Disturbingly Hilarious Pr...
6,5,272,5_women_men_feminist_to,"[women, men, feminist, to, woman, feminism, yo...",[Here Are Some Posters For The Men Who Tell Wo...
7,6,188,6_science_doctor_the_to,"[science, doctor, the, to, you, of, all, facts...","[16 Years Ago, A Doctor Published A Study. It ..."
8,7,181,7_cat_dog_animals_dogs,"[cat, dog, animals, dogs, elephants, to, the, ...","[You've Probably Seen Tons Of Cat Videos, But ..."
9,8,180,8_song_music_the_of,"[song, music, the, of, it, to, that, this, rap...",[One Of The Best Parts Of This Song Is That It...


Looking at the most frequent words in the most common topics we easily interpret what most of them are about:

- –1: Miscellaneous
- 0: She, He, Pronouns
- 1: Kids
- 2: Food & Restaurants
- 3: Racism
- 4: Gay Marriage
- 5: Feminism
- 6: Science & Medicine
- 7: Pets & Animals
- 8: Music
- 9: Crime
- 10: Water & City
- 11: Guns & Wars
- 12: Money & Economy
- 13: Fashion
- 14: Climate Change
- 15: Space
- 16: Viral Videos & Content
- 17: Men (He, Him, His)
- 18: Future & Predictions

Now I'll save the dataset

In [11]:
exploratory_df.to_csv("dataset/processed/exploratory-packages-topic-modeling.csv", index=False)
confirmatory_df.to_csv("dataset/processed/confirmatory-packages-topic-modeling.csv", index=False)
holdout_df.to_csv("dataset/processed/holdout-packages-topic-modeling.csv", index=False)

For future reference, I'll save all the topics with the most frequent words.

In [12]:
# Save the topics
topics = topic_model.get_topics()

rows = []
for topic_id, word_weights in topics.items():
    # Take the top 50 words for this topic
    top_words = [word for word, _ in word_weights[:50]]
    rows.append({
        "topic_id": topic_id,
        "top_50_words": ", ".join(top_words),
    })

topics_df = pd.DataFrame(rows).sort_values("topic_id")

output_path = "dataset/processed/topics.csv"
topics_df.to_csv(output_path, index=False)
topics_df.head()

,topic_id,top_50_words
0,-1,"the, to, you, of, and, it, this, in, is, that,..."
1,0,"she, her, to, the, it, and, was, in, mom, but,..."
2,1,"kids, to, this, the, teachers, these, kid, of,..."
3,2,"food, eat, you, it, the, that, to, fast food, ..."
4,3,"black, white, racist, race, people, racism, it..."
